# The EM algorithms behind EMU and PCAngsd

A PCA of genotypes looks like a one-shot calculation: normalize the genotype matrix, take the
singular value decomposition, plot the first two components. That works as long as you actually
*have* the genotypes. Two situations break it, and both are the rule rather than the exception
in modern data sets

 - **missing data.** Genotype chips and low coverage data have missing genotypes everywhere.
   Almost all PCA software (`SmartPCA`/`eigensoft`, `plink2 --pca`, `flashPCA`, ...) fills a
   missing genotype with the mean genotype at that SNP. At 90-99% missingness that is mostly
   made-up data, and the PCA is dominated by it.
 - **low depth sequencing.** Here *every* genotype is uncertain. Calling the genotype and
   pretending it is known throws away that uncertainty and biases the result.

`EMU` and `PCAngsd` solve the two problems with the same idea, which is the one on the slides

> If we have the PCA we can calculate the individual allele frequencies.
> If we have the individual allele frequencies we can calculate the PCA.

That is a chicken-and-egg problem, and the way out is the one you have used for the admixture
model and for the SFS: guess, then iterate. Fill in the unknown part of the data with its
**expected value** given the current parameters, re-estimate the parameters from the completed
data, and repeat. That is the structure of an EM algorithm - the E-step is the expected
genotype, the M-step is the low-rank PCA.

**Learning objectives**

 - Be able to write the PCA as a matrix decomposition and read the individual allele
   frequencies out of it
 - See why mean imputation distorts a PCA when there is a lot of missing data
 - Implement the EMU iteration and watch it recover the true structure
 - Understand why the posterior genotype (the dosage) is the right thing to put in the matrix
   for low depth data
 - Implement the PCAngsd iteration and compare it with calling genotypes

# 1. PCA, low rank approximation and individual allele frequencies

Let $G$ be the $N \times M$ matrix of genotypes (individuals by SNPs) with $G_{ij}\in\{0,1,2\}$
counting the number of minor alleles, and let $f_j$ be the allele frequency at SNP $j$. The
normalized genotype matrix is

$$ M_{ij}=\frac{G_{ij}-2f_j}{\sqrt{2f_j(1-f_j)}} $$

Every SNP now has mean zero and roughly variance one, so common and rare SNPs contribute
equally. The PCA is the singular value decomposition of that matrix

$$ M = U S V^{T} $$

where the columns of $U$ are the principal components of the individuals, $S$ is diagonal with
the singular values, and $V$ holds the SNP weights.

**The key step.** If we keep only the first $k$ components we get the *low rank approximation*
of the data

$$ \hat{M} = U_{[1:k]}S_{[1:k]}V^{T}_{[1:k]} $$

Undoing the normalization turns this into a matrix of **individual allele frequencies** (the
slide writes it as $\Pi$)

$$ \frac{1}{2}\mathbb{E}[G] \approx \Pi = U_{[1:k]}S_{[1:k]}V^{T}_{[1:k]} \quad\text{(after undoing the scaling)} $$

$\Pi_{ij}$ is the frequency of the allele at SNP $j$ *for individual $i$* - the same quantity
you met in the admixture model, where it was $\sum_k Q_{ik}F_{jk}$. The admixture model gets it
from $K$ discrete populations; PCA gets it from $k$ continuous axes of variation. This is the
bridge that makes the whole exercise work: **a PCA predicts a genotype.**

**Questions**

 - Why do we divide by $\sqrt{2f_j(1-f_j)}$? What is that number for a binomial with 2 trials?
 - The full SVD reproduces $M$ exactly. What do we throw away by keeping only $k$ components,
   and why is that a good thing here?
 - $\Pi_{ij}$ is an allele frequency, so it must be between 0 and 1. Is that guaranteed by the
   formula above?

In [ ]:
## ---------------------------------------------------------------------------
## the three building blocks we will use over and over
## ---------------------------------------------------------------------------

## normalize the genotypes: subtract 2f and divide by the binomial standard deviation
normalize <- function(X, f){
  Mn <- sweep(X,  2, 2*f, "-")
  sweep(Mn, 2, sqrt(2*f*(1-f)), "/")
}

## the low rank approximation: keep only the k first components of the SVD
truncSVD <- function(Mn, k){
  s <- svd(Mn, nu = k, nv = k)
  s$u %*% diag(s$d[1:k], k, k) %*% t(s$v)
}

## the principal components of the individuals, scaled by the singular values
getPC <- function(X, f, k = 2){
  s <- svd(normalize(X, f))
  s$u[,1:k] %*% diag(s$d[1:k])
}

## go from a low rank approximation of the NORMALIZED data back to the genotype scale,
## i.e. 2 * Pi  - the expected genotype of each individual at each SNP
predictG <- function(Mhat, f){
  P <- sweep(sweep(Mhat, 2, sqrt(2*f*(1-f)), "*"), 2, 2*f, "+")
  pmin(pmax(P, 0), 2)          # an expected genotype has to stay between 0 and 2
}

**Questions**
 - `normalize()` divides by $\sqrt{2f(1-f)}$, the binomial standard deviation. What would happen to a rare SNP if we did not divide by it?
 - `truncSVD` keeps only the first $k$ components. What is thrown away, and why is that the point?

# 2. Simulated data

We simulate three populations with 20 individuals each. The populations are only weakly
differentiated ($F_{ST}=0.05$), so the structure is real but not trivial to see - as in real
human data.

In [ ]:
set.seed(3)
N <- 60      # individuals, 20 from each population
M <- 10000   # SNPs
Fst <- 0.05  # differentiation between the three populations

anc <- runif(M, 0.1, 0.9)
bn  <- function(p, Fst) rbeta(length(p), p*(1-Fst)/Fst, (1-p)*(1-Fst)/Fst)
Ftrue <- cbind(bn(anc,Fst), bn(anc,Fst), bn(anc,Fst))    # allele freq in each population
Qtrue <- rbind(matrix(rep(c(1,0,0), each=20), 20),       # who belongs to which population
               matrix(rep(c(0,1,0), each=20), 20),
               matrix(rep(c(0,0,1), each=20), 20))

G   <- matrix(rbinom(N*M, 2, Qtrue %*% t(Ftrue)), N, M)  # the true genotypes
pop <- rep(1:3, each = 20)
palette(c("steelblue","darkorange","forestgreen"))

cat("genotype matrix:", dim(G), "\n")
print(G[1:5, 1:10])

**Questions**
 - $F_{ST}=0.05$ is weak differentiation. Would the exercise be more or less convincing with $F_{ST}=0.30$?
 - There are 60 individuals and 10000 SNPs. Which of the two numbers limits how well the PCA can resolve the populations?

This is the PCA we are trying to recover: the one we would get if we knew every genotype.

In [ ]:
f_all  <- colMeans(G)/2
PCtrue <- getPC(G, f_all)

plotPC <- function(PC, main){
  plot(PC[,1], PC[,2], col = pop, pch = 16, cex = 1.3,
       xlab = "PC1", ylab = "PC2", main = main)
}
plotPC(PCtrue, "TRUE PCA (all genotypes known)")
legend("topright", fill = 1:3, c("pop 1","pop 2","pop 3"), bty = "n")

**Question**
 - This is the PCA we are trying to recover. How clearly are the three populations separated, and along which components?

To say how good an estimated PCA is we need a number. Comparing PC1 with PC1 directly does not
work: the sign of a principal component is arbitrary, and two PCs that explain almost the same
amount of variance can swap places or be rotated into each other. What *is* meaningful is
whether the individuals end up in the same relative positions, so we compare all the pairwise
distances in the PC plot

$$ \text{dcor}=\text{cor}\Big(\; d^{\text{est}}_{ab}\;,\; d^{\text{true}}_{ab} \;\Big),
\qquad d_{ab}=\sqrt{(\text{PC1}_a-\text{PC1}_b)^2+(\text{PC2}_a-\text{PC2}_b)^2} $$

over all pairs of individuals $a,b$. It is 1 when the two plots place the individuals in the
same configuration, no matter how the plot is rotated or flipped.

In [ ]:
## 1 means the individuals sit exactly as they do in the true PCA, 0 means no resemblance
dcor <- function(A, B) cor(as.vector(dist(A[,1:2])), as.vector(dist(B[,1:2])))

cat("sanity check, the true PCA compared with itself:", dcor(PCtrue, PCtrue), "\n")

**Question**
 - Why compare *distances between individuals* rather than PC1 with PC1 directly?

# 3. Missing data and mean imputation

Now we throw genotypes away. Importantly the individuals are **not** equally affected: some
have half their genotypes left, others have almost nothing. That is what real data looks like,
and it is what makes mean imputation dangerous.

In [ ]:
indMiss <- runif(N, 0.05, 0.90)                # each individual has its own missingness
Gm <- G
Gm[matrix(runif(N*M) < indMiss, N, M)] <- NA

cat("overall fraction missing :", round(mean(is.na(Gm)), 3), "\n")
cat("per individual           :", round(range(rowMeans(is.na(Gm))), 2), "\n")

## the allele frequency is estimated from whatever is observed
f <- colMeans(Gm, na.rm = TRUE)/2
ok <- !is.na(f) & f > 0.05 & f < 0.95          # drop SNPs with too little information
Gm <- Gm[, ok]; Gsub <- G[, ok]; f <- f[ok]
cat("SNPs left                :", ncol(Gm), "\n")

## the PCA we should be aiming for, using the same SNPs
PCtrue <- getPC(Gsub, colMeans(Gsub)/2)

**Questions**
 - Each individual has its own missingness, between 5% and 90%. Why does that matter more than the overall fraction missing?
 - Is the missingness related to which population an individual comes from?

Mean imputation is what nearly all PCA software does: a missing genotype is replaced by twice
the allele frequency, which is the mean genotype at that SNP

$$ G_{ij} \;\leftarrow\; \mathbb{E}[G_{ij}] = 2f_j \qquad \text{for every missing entry} $$

After normalization that is the same as setting the missing entries to **zero**, because
$M_{ij}=(2f_j-2f_j)/\sqrt{2f_j(1-f_j)}=0$. Every individual is therefore pushed towards the
average individual by exactly as much data as it is missing.

In [ ]:
meanImpute <- function(Gm, f){
  E <- Gm
  for(j in 1:ncol(Gm))
    E[is.na(Gm[,j]), j] <- 2*f[j]
  E
}

Emean  <- meanImpute(Gm, f)
PCmean <- getPC(Emean, f)

par(mfrow = c(1,2))
plotPC(PCtrue, "TRUE PCA")
plotPC(PCmean, "mean imputation")
cat("distance correlation with the true PCA:", round(dcor(PCmean, PCtrue), 3), "\n")

 - Compare the two plots. Are the three populations still separated?
 - An individual with 99% missing data has 99% of its row set to the same value as everybody
   else's missing entries. Where in the plot do you expect it to end up, and why?
 - Try `plot(rowMeans(is.na(Gm)), sqrt(rowSums(PCmean[,1:2]^2)))`. What does the distance from
   the centre of the plot depend on?
 - Is this a problem with the PCA, or with the data we fed into it?

# 4. EMU

EMU fixes this by not treating the imputed values as data. The algorithm is exactly the picture
on the slides

 1. **start**: fill the missing genotypes with the mean, $2f_j$
 2. **M-step**: do the PCA of the completed matrix and keep only the first $k$ components -
    this is the low rank approximation $\hat M = U_{[1:k]}S_{[1:k]}V^{T}_{[1:k]}$
 3. **E-step**: the low rank approximation predicts a genotype for *every* entry of the matrix,
    $2\Pi_{ij}$. Replace the missing entries - and only those - with that prediction
 4. repeat 2 and 3

The observed genotypes never change: they are data. Only the holes are refilled, each time with
a better guess, because each round the prediction comes from a PCA that was itself computed
from better filled-in data.

In [ ]:
emu <- function(Gm, k = 2, iter = 10){
  miss <- is.na(Gm)
  f <- colMeans(Gm, na.rm = TRUE)/2

  E <- meanImpute(Gm, f)                 # step 1: start from mean imputation
  for(it in 1:iter){
    Mhat <- truncSVD(normalize(E, f), k) # step 2: low rank PCA of the completed matrix
    Ghat <- predictG(Mhat, f)            # step 3: what the PCA predicts for every genotype
    E[miss] <- Ghat[miss]                #         put the prediction back, only in the holes
  }
  E
}

## how good is the PCA after 1, 5, 20 and 50 rounds?
for(it in c(1, 5, 20, 50)){
  PCe <- getPC(emu(Gm, k = 2, iter = it), f)
  cat("EMU with", formatC(it, width=3), "iterations :", round(dcor(PCe, PCtrue), 3), "\n")
}
cat("mean imputation (0 iterations) :", round(dcor(PCmean, PCtrue), 3), "\n")

**Questions**
 - Find the E-step and the M-step in this function. Which line is which?
 - The loop starts from mean imputation. Does the starting point matter for where it ends up?

In [ ]:
PCemu <- getPC(emu(Gm, k = 2, iter = 50), f)

par(mfrow = c(1,3))
plotPC(PCtrue, "TRUE PCA")
plotPC(PCmean, "mean imputation")
plotPC(PCemu,  "EMU, 50 iterations")

**Question**
 - Compare the three plots. Has EMU recovered the separation that mean imputation lost?

## Colour the individuals by how much data they are missing

The plots above are coloured by population, which is the answer we are trying to find. Let us
instead colour every individual by the fraction of its genotypes that is missing - something we
always know, also for real data where we do not know the populations.

In [ ]:
missInd <- rowMeans(is.na(Gm))          # fraction missing for each individual

## dark blue = almost complete data, red = almost everything missing
missCol <- function(m){
  ramp <- colorRampPalette(c("navy", "steelblue", "gold", "red"))(100)
  ramp[ cut(m, breaks = seq(0, 1, length.out = 101), labels = FALSE) ]
}
cols <- missCol(missInd)

par(mfrow = c(1,3), mar = c(4,4,3,1))
plot(PCtrue[,1], PCtrue[,2], col = cols, pch = 16, cex = 1.6,
     xlab = "PC1", ylab = "PC2", main = "TRUE PCA")
legend("topright", pch = 16, col = missCol(c(0.1, 0.4, 0.7, 0.9)),
       legend = paste0(c(10, 40, 70, 90), "% missing"), bty = "n", cex = 1.1)
plot(PCmean[,1], PCmean[,2], col = cols, pch = 16, cex = 1.6,
     xlab = "PC1", ylab = "PC2", main = "mean imputation")
plot(PCemu[,1], PCemu[,2], col = cols, pch = 16, cex = 1.6,
     xlab = "PC1", ylab = "PC2", main = "EMU, 50 iterations")

 - In the mean imputation plot, where do the red individuals sit compared with the dark blue
   ones? Is that biology or is it an artefact?
 - In the true PCA the colours are scattered at random, because missingness has nothing to do
   with which population an individual comes from. Is that also the case after EMU?
 - How far the imputed values pull an individual is measured by its distance from the centre
   of the plot. Compare the two approaches directly:

```
plot(missInd, sqrt(rowSums(PCmean[,1:2]^2)), xlab = "fraction missing", ylab = "distance from origin")
points(missInd, sqrt(rowSums(PCemu[,1:2]^2)), col = 2, pch = 16)
```

 - Imagine you did not have the true PCA to compare with - only the plot coloured by
   missingness. Would that be enough to tell you something had gone wrong?

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/em_algorithms/quiz/pca_em_missing.json")


 - Follow the numbers as the iterations increase. Does each round help?
 - Find the E-step and the M-step in the function. Which line is which?
 - `E[miss] <- Ghat[miss]` only touches the missing entries. What would go wrong if we wrote
   `E <- Ghat` and overwrote everything?
 - The algorithm needs the number of components $k$ up front. Try `k = 1` and `k = 5`. What
   happens when $k$ is too small, and what happens when it is too large?
 - The allele frequency `f` is estimated once from the observed genotypes and then never
   updated. Do you think that is a reasonable approximation?

# 5. PCAngsd - the same idea for low depth sequencing

With low depth data nothing is flatly missing, but nothing is certain either: a site covered by
one read tells us something about the genotype, just not everything. The information is in the
**genotype likelihoods** $p(X_{ij}\mid G_{ij}=g)$ for $g=0,1,2$.

We simulate sequencing of the same individuals. As in a real study the individuals are **not**
sequenced equally deeply - here the average depth of an individual is somewhere between 0.3X and
5X. The depth of individual $i$ at site $j$ is Poisson distributed around that individual's
average, $D_{ij}\sim\text{Poisson}(\bar D_i)$, each read shows the minor allele with a
probability that depends on the genotype, and with probability $\epsilon$ a read is wrong

| genotype $g$ | $p(\text{read shows the minor allele}\mid g)$ |
|---|---|
| 0 | $\epsilon$ |
| 1 | $0.5$ |
| 2 | $1-\epsilon$ |

so with $k_{ij}$ minor allele reads out of $D_{ij}$ the genotype likelihood is binomial

$$ p(X_{ij}\mid G_{ij}=g)=\binom{D_{ij}}{k_{ij}}\,p_g^{\,k_{ij}}\,(1-p_g)^{D_{ij}-k_{ij}} $$

In [ ]:
simGL <- function(G, depth, e = 0.01){
  N <- nrow(G); M <- ncol(G)
  d  <- matrix(rpois(N*M, rep(depth, M)), N, M)  # number of reads, depth[i] per individual
  pd <- c(e, 0.5, 1-e)[G+1]                      # p(read shows the minor allele | genotype)
  k  <- matrix(rbinom(N*M, d, pd), N, M)         # reads with the minor allele
  list(g0 = dbinom(k, d, e), g1 = dbinom(k, d, 0.5), g2 = dbinom(k, d, 1-e), depth = d)
}

set.seed(7)
indDepth <- runif(N, 0.3, 5)                     # each individual has its own average depth
GL <- simGL(G, depth = indDepth)

cat("average depth over everybody   :", round(mean(GL$depth), 2), "\n")
cat("per individual                 :", round(range(rowMeans(GL$depth)), 2), "\n")
cat("fraction of sites with no reads:", round(mean(GL$depth == 0), 3), "\n")

**Questions**
 - The depth is Poisson with a different mean per individual. What does that produce that a constant depth would not?
 - With an error rate of 1%, how much does a single read tell you about the genotype?

The allele frequency has to be estimated from the likelihoods as well, with the EM algorithm you
already know from the genotype calling exercise. The hidden variable is the genotype and the
parameter is the frequency $f_j$.

**E-step** - the posterior probability of each genotype, with Hardy-Weinberg as the prior

$$ p(G_{ij}=g\mid X_{ij},f_j)=\frac{p(X_{ij}\mid G_{ij}=g)\,p(G_{ij}=g\mid f_j)}
{\sum_{g'=0}^{2}p(X_{ij}\mid G_{ij}=g')\,p(G_{ij}=g'\mid f_j)}
\qquad
p(G=g\mid f)=\binom{2}{g}f^{g}(1-f)^{2-g}$$

**M-step** - the new frequency is the expected number of minor alleles divided by the number of
alleles

$$ f_j^{(n+1)}=\frac{1}{2N}\sum_{i=1}^{N}\mathbb{E}[G_{ij}\mid X_{ij},f_j^{(n)}]
=\frac{1}{2N}\sum_{i=1}^{N}\sum_{g=0}^{2} g\; p(G_{ij}=g\mid X_{ij},f_j^{(n)}) $$

In [ ]:
estFreq <- function(GL, iter = 100){
  f <- rep(0.5, ncol(GL$g0))
  for(i in 1:iter){
    p0 <- sweep(GL$g0, 2, (1-f)^2,    "*")       # p(X|G=g) p(G=g|f) under HWE
    p1 <- sweep(GL$g1, 2, 2*f*(1-f),  "*")
    p2 <- sweep(GL$g2, 2, f^2,        "*")
    post <- (p1 + 2*p2) / (p0 + p1 + p2)         # expected number of minor alleles
    f <- colMeans(post)/2
  }
  f
}

fgl <- estFreq(GL)
ok  <- fgl > 0.05 & fgl < 0.95
GL  <- list(g0 = GL$g0[,ok], g1 = GL$g1[,ok], g2 = GL$g2[,ok], depth = GL$depth[,ok])
Gs  <- G[,ok]; fgl <- fgl[ok]
PCtrueGL <- getPC(Gs, colMeans(Gs)/2)

cat("SNPs left:", ncol(Gs), "  correlation with the true frequencies:",
    round(cor(fgl, colMeans(Gs)/2), 3), "\n")

**Questions**
 - This is the same EM you met in the genotype-calling exercise. What is the hidden variable and what is the parameter?
 - Why must the frequency be estimated from the likelihoods rather than from called genotypes?

## The wrong way: call the genotypes first

The obvious thing is to take the most likely genotype and run a normal PCA

$$ \widehat{G}_{ij}=\arg\max_{g\in\{0,1,2\}}\; p(X_{ij}\mid G_{ij}=g) $$

Note what this does at a site with no reads: all three likelihoods are equal to 1, and the call
is whatever comes first - a genotype invented out of nothing.

In [ ]:
callGeno <- function(GL){
  cl <- max.col(cbind(as.vector(GL$g0), as.vector(GL$g1), as.vector(GL$g2))) - 1
  matrix(cl, nrow(GL$g0), ncol(GL$g0))
}

Gcall  <- callGeno(GL)
PCcall <- getPC(Gcall, colMeans(Gcall)/2)
cat("called genotypes, distance correlation:", round(dcor(PCcall, PCtrueGL), 3), "\n")

**Question**
 - At a site with no reads all three likelihoods are equal. What genotype does `max.col` return there, and is that information or noise?

## The dosage

Instead of committing to one genotype we use the **expected** genotype. Bayes' formula turns the
genotype likelihood into a posterior probability, using the allele frequency as the prior
(slide *PCA for low depth sequencing*)

$$ p(G=g\mid X,\pi)=\frac{p(X\mid G=g)\,p(G=g\mid \pi)}{\sum_{g'=0}^{2}p(X\mid G=g')\,p(G=g'\mid \pi)} $$

where $p(G=g\mid\pi)$ is the Hardy-Weinberg probability of the genotype given the allele
frequency $\pi$. The expected genotype - the **dosage** - is

$$ \mathbb{E}[G\mid X,\pi]=\sum_{g=0}^{2} g\; p(G=g\mid X,\pi) $$

and that is the number we put in the matrix instead of the genotype. Note what happens at a site
with no reads: all three likelihoods are equal, the posterior collapses to the HWE prior, and
the dosage becomes $2\pi$ - exactly mean imputation. **Low depth data is missing data with
shades of grey.**

In [ ]:
dosage <- function(GL, Pi){
  p0 <- GL$g0 * (1-Pi)^2                        # p(X|G=0) p(G=0|Pi)
  p1 <- GL$g1 * 2*Pi*(1-Pi)
  p2 <- GL$g2 * Pi^2
  (p1 + 2*p2) / (p0 + p1 + p2)                  # 0*p0 + 1*p1 + 2*p2, normalized
}

## first try: use the population allele frequency as the prior for everybody
Pi0 <- matrix(fgl, N, ncol(Gs), byrow = TRUE)
D0  <- dosage(GL, Pi0)
PCd0 <- getPC(D0, fgl)
cat("dosage with the population frequency:", round(dcor(PCd0, PCtrueGL), 3), "\n")

 - Compare the two numbers. Is the dosage with a population prior actually better than calling?
 - Think about what the prior does to a **shallowly sequenced** individual. At a site with no
   reads the dosage is exactly $2f_j$ - the population mean. That is mean imputation again, and
   it pulls the low depth individuals towards the centre of the plot, exactly like in the EMU
   section. Calling genotypes does not shrink anybody, it just adds noise. So we have traded one
   problem for another.
 - The prior we used is the same for every individual. For an individual from population 3 at a
   SNP that is common in population 1 and rare in population 3, is that prior right?

## PCAngsd: an individual prior

That last question is the whole point. The prior should not be the *population* allele
frequency, it should be the **individual** allele frequency $\Pi_{ij}$ - and we know how to get
those from a PCA. So we iterate, exactly as in EMU

 1. **start**: $\Pi_{ij}=f_j$ for everybody
 2. **E-step**: compute the dosages $\mathbb{E}[G\mid X,\Pi]$ with the current $\Pi$ as the prior
 3. **M-step**: PCA of the dosage matrix, keep $k$ components, and read off the new individual
    allele frequencies $\Pi = \frac{1}{2}\,U_{[1:k]}S_{[1:k]}V^{T}_{[1:k]}$ (back on the
    genotype scale)
 4. repeat 2 and 3

In [ ]:
pcangsd <- function(GL, f, k = 2, iter = 10){
  N <- nrow(GL$g0); M <- ncol(GL$g0)
  Pi <- matrix(f, N, M, byrow = TRUE)           # step 1: everybody gets the population freq
  for(it in 1:iter){
    D    <- dosage(GL, Pi)                      # step 2 (E): expected genotypes
    Mhat <- truncSVD(normalize(D, f), k)        # step 3 (M): low rank PCA
    Pi   <- predictG(Mhat, f) / 2               #             individual allele frequencies
    Pi   <- pmin(pmax(Pi, 1e-4), 1 - 1e-4)      #             keep them away from 0 and 1
  }
  dosage(GL, Pi)                                # the final dosage matrix
}

for(it in c(1, 3, 10, 30)){
  PCp <- getPC(pcangsd(GL, fgl, k = 2, iter = it), fgl)
  cat("PCAngsd with", formatC(it, width=2), "iterations :", round(dcor(PCp, PCtrueGL), 3), "\n")
}

**Questions**
 - Line by line, which part is the E-step and which the M-step?
 - `Pi` starts as the population frequency for everybody and becomes individual-specific. That is the whole idea — why does it help the shallow individuals most?

In [ ]:
PCpc <- getPC(pcangsd(GL, fgl, k = 2, iter = 30), fgl)

par(mfrow = c(2,2))
plotPC(PCtrueGL, "TRUE PCA (known genotypes)")
plotPC(PCcall,   "called genotypes")
plotPC(PCd0,     "dosage, population frequency")
plotPC(PCpc,     "PCAngsd, 30 iterations")

**Question**
 - Put the four panels in order of how well they recover the true PCA. Which single change buys the most?

Just as we did for the missing data, let us colour the individuals by how well they were
sequenced instead of by population. Here dark blue is a deeply sequenced individual and red is a
shallow one.

In [ ]:
depthInd <- rowMeans(GL$depth)
depthCol <- function(d){
  ramp <- colorRampPalette(c("red", "gold", "steelblue", "navy"))(100)
  ramp[ cut(d, breaks = seq(0, max(d), length.out = 101), labels = FALSE) ]
}
dcols <- depthCol(depthInd)

par(mfrow = c(1,3), mar = c(4,4,3,1))
plot(PCcall[,1], PCcall[,2], col = dcols, pch = 16, cex = 1.6,
     xlab = "PC1", ylab = "PC2", main = "called genotypes")
legend("topright", pch = 16, col = depthCol(c(0.5, 2, 3.5, 5)),
       legend = paste0(c(0.5, 2, 3.5, 5), "X"), bty = "n", cex = 1.1)
plot(PCd0[,1], PCd0[,2], col = dcols, pch = 16, cex = 1.6,
     xlab = "PC1", ylab = "PC2", main = "dosage, population frequency")
plot(PCpc[,1], PCpc[,2], col = dcols, pch = 16, cex = 1.6,
     xlab = "PC1", ylab = "PC2", main = "PCAngsd, 30 iterations")

 - Where do the red (shallow) individuals sit in each of the three plots?
 - Which plot would make you believe that sequencing depth is a population of its own?
 - After PCAngsd, can you tell from the plot which individuals were sequenced deeply?

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/em_algorithms/quiz/pca_em_lowdepth.json")


# 6. Randomized SVD: Halko and PCAone

Both algorithms above call `svd()` inside their loop. That is fine for 60 individuals and 10000
SNPs, but real data is 100000 individuals by 10 million SNPs, the matrix does not fit in memory,
and a full SVD is out of the question. We also do not *want* the full SVD: we throw away
everything but the first $k$ components anyway.

A **randomized SVD** (RSVD) computes exactly those first $k$ components, and it does it with the
same kind of iteration you have now seen twice. We will first build the standard version - the
**Halko** algorithm, which `PCAone` also implements - and then the window based scheme that
`PCAone` uses by default.

## The idea: look at the data through a small random window

Multiply the $N\times M$ normalized genotype matrix $A$ by a random matrix
$\Omega$ with only $l=k+p$ columns ($p$ is a small amount of oversampling)

$$ Y = A\,\Omega \qquad (N\times l) $$

Each column of $Y$ is a random mixture of all the SNPs. The directions in which the data varies
the most survive that mixing; the noise directions largely cancel. So the column space of $Y$ -
a space of only $l$ dimensions instead of $M$ - already contains most of what we are looking for.
Orthonormalize it to get a basis $Q$, and then the small $l \times M$ matrix

$$ B = Q^{T}A $$

is a compressed version of the data that we *can* take the SVD of. If $B=\tilde U S V^{T}$ then
$U = Q\tilde U$ and we have our PCA.

## Why one random draw is not enough - the Halko power iteration

If the singular values are well separated, one draw is fine. With genetic data they are not:
population structure gives several components of almost the same size, and a random mixture
cannot tell them apart in one go. The fix is to stop using a *random* $\Omega$ and start using a
*good* one - and the best test matrix we could possibly use would be the true right singular
vectors themselves.

We do not have those, but we have an estimate, and we can improve it. That gives the loop

 1. **start**: $\Omega$ is random, $Q=\text{orth}(A\Omega)$
 2. **E-step** - look at the data through the current subspace: $\Omega \leftarrow \text{orth}(A^{T}Q)$
 3. **M-step** - re-estimate the subspace from it: $Q \leftarrow \text{orth}(A\,\Omega)$
 4. repeat

$\Omega$ is updated in every iteration, and each update makes it point more towards the true
singular vectors. This is the same shape of algorithm as EMU and PCAngsd: the unknown here is
not a genotype but the *subspace*, and each round we project the data onto our current guess and
re-estimate the guess from the projection.

This is the algorithm of Halko et al., and it is what `PCAone` runs with `--svd 2`. Keep track of
one thing while you read the code: **each update of $\Omega$ touches the whole data matrix
twice**, once for $A^{T}Q$ and once for $A\Omega$. For a matrix that does not fit in memory that
means two full reads from disk per update, and reading from disk - not the arithmetic - is what
the whole thing costs.

In [ ]:
halko <- function(A, k = 2, l = k + 5, iter = 5){
  Omega <- matrix(rnorm(ncol(A)*l), ncol(A), l)   # start with a random test matrix
  Q <- qr.Q(qr(A %*% Omega))                      # first sketch of the subspace

  for(it in seq_len(iter)){
    Omega <- qr.Q(qr(t(A) %*% Q))                 # update Omega towards the current subspace
    Q     <- qr.Q(qr(A %*% Omega))                # project the data onto it again
  }

  B <- t(Q) %*% A                                 # the compressed data, only l x M
  s <- svd(B)                                     # a small SVD we can actually afford
  list(u = Q %*% s$u[,1:k, drop=FALSE], d = s$d[1:k], v = s$v[,1:k, drop=FALSE])
}

**Questions**
 - `Omega` is random. Why does multiplying by a random matrix capture the important directions at all?
 - Each `iter` does two matrix products with `A`. In out-of-core terms, what does that cost?

Let us run it on the normalized genotypes and compare with the exact answer from `svd()`.

In [ ]:
A <- normalize(G, f_all)                          # the full normalized genotype matrix
full <- svd(A)
PCfull <- full$u[,1:2] %*% diag(full$d[1:2])
cat("exact singular values :", round(full$d[1:2], 2), "\n\n")

for(it in c(0, 1, 2, 5, 10)){
  set.seed(1)
  r  <- halko(A, k = 2, iter = it)
  PC <- r$u %*% diag(r$d)
  cat("Omega updated", formatC(it, width=2), "times (", formatC(2*it+1, width=2),
      "reads of the data ): singular values",
      formatC(round(r$d, 2), width=6),
      "  dcor with the exact PCA:", round(dcor(PC, PCfull), 4), "\n")
}

In [ ]:
set.seed(1)
r0 <- halko(A, k = 2, iter = 0)                  # no update: just one random sketch
set.seed(1)
r5 <- halko(A, k = 2, iter = 5)

par(mfrow = c(1,3))
plotPC(PCfull,                      "exact svd()")
plotPC(r0$u %*% diag(r0$d),         "Halko, 0 updates of Omega")
plotPC(r5$u %*% diag(r5$d),         "Halko, 5 updates of Omega")

 - How many updates of $\Omega$ does it take before the singular values match the exact ones,
   and how many reads of the data is that?
 - With 0 updates the plot is a random projection of the data. Can you still see three groups?
 - $\Omega$ has $l=k+5$ columns rather than $k$. Why is it worth carrying a few extra
   dimensions? Try `l = k` and see.
 - The only place the big matrix $A$ appears is in the products $A\Omega$ and $A^{T}Q$. Both can
   be computed by reading $A$ one chunk of SNPs at a time and adding up the pieces. Why does
   that matter for a data set that does not fit in memory?
 - `PCAone` and `PCAngsd` are often used together: `PCAngsd` needs a truncated SVD in every one
   of its iterations. Which of the two loops sits inside the other?

## What PCAone actually does: the window

The Halko loop above has one weakness, and it is not accuracy - it is **reads of the data**. Each
update of $\Omega$ uses the whole matrix, so out-of-core, where the matrix is on disk, a handful
of updates means a handful of full passes over hundreds of gigabytes.

`PCAone` breaks that link. The observation is that $\Omega$ can be improved using only *part* of
the data: the product $A^{T}Q$ is a sum of contributions from the SNPs, so a block of SNPs
already gives a usable, if noisy, direction to move in. Instead of reading everything and then
taking one big step, take many small steps while reading the data once. This is the same idea as
stochastic gradient descent, applied to a subspace instead of a parameter.

The scheme in the paper (Li, Meisner & Albrechtsen 2023) is

 1. **permute the SNPs** and split the matrix into $b=64$ blocks. The permutation matters: SNPs
    that sit next to each other are in linkage disequilibrium, so a block of consecutive SNPs
    would not be a representative subset of the genome
 2. **epoch 1**: slide a window of 2 blocks across the data one block at a time - blocks 1-2,
    then 2-3, then 3-4 and so on. At every window position, accumulate $H \mathrel{+}= A_{W}A_{W}^{T}Q$
    from the SNPs in the window and update $Q=\text{orth}(H)$. One sweep across the data,
    but $\Omega$ (equivalently $Q$) has been updated 63 times instead of once
 3. **double the window and the step** after every epoch: 4 blocks stepping 2, then 8 stepping
    4, and so on
 4. by epoch 6 the window is the whole matrix, so the last epochs are exactly the ordinary power
    iterations we did above
 5. stop when the eigenvectors stop changing between epochs ($1-\text{MEV}<10^{-4}$), which by
    default happens at 7 epochs

In the first five epochs `PCAone` updates $\Omega$ over a hundred times (the paper counts 124)
where plain power iterations manage 5.

In [ ]:
winsvd <- function(A, k = 2, l = k + 5, epochs = 7, blocks = 64){
  N <- nrow(A); M <- ncol(A)

  A   <- A[, sample(M)]                              # permute the SNPs: no LD within a block
  bnd <- round(seq(0, M, length.out = blocks + 1))   # the block boundaries

  Q <- qr.Q(qr(A %*% matrix(rnorm(M*l), M, l)))      # the same random start as before
  w <- 2                                             # window size, in blocks
  nUpdate <- 0

  for(e in seq_len(epochs)){
    if(w >= blocks){                                 # the window is the whole matrix:
      G <- t(A) %*% Q                                # an ordinary power iteration
      Q <- qr.Q(qr(A %*% G))
      nUpdate <- nUpdate + 1
    } else {
      H <- matrix(0, N, l)                           # accumulated over this sweep
      for(start in seq(1, blocks - w + 1, by = w %/% 2)){     # slide the window
        idx <- (bnd[start] + 1):bnd[start + w]               # the SNPs in the window
        G   <- t(A[, idx, drop=FALSE]) %*% Q                 # what this window says
        H   <- H + A[, idx, drop=FALSE] %*% G                # add it to the running sum
        Q   <- qr.Q(qr(H))                                   # and update straight away
        nUpdate <- nUpdate + 1
      }
      w <- w * 2                                     # double the window and the step
    }
  }
  s <- svd(t(Q) %*% A)
  list(u = Q %*% s$u[,1:k, drop=FALSE], d = s$d[1:k], updates = nUpdate)
}

**Questions**
 - The SNPs are permuted before being split into blocks. What would go wrong without that?
 - `Q` is updated once per block rather than once per pass. How many updates does one epoch give?

Now compare the two on equal terms. The fair currency is **passes over the data**: one Halko
power iteration costs two, one window epoch costs one.

In [ ]:
cat("exact singular values:", round(full$d[1:2], 2), "\n\n")

cat("Halko\n")
for(it in 1:6){
  set.seed(1); r <- halko(A, k = 2, iter = it)
  cat("  ", it, "power iterations (", formatC(2*it, width=2), "passes,",
      formatC(it, width=3), "updates ): dcor", round(dcor(r$u %*% diag(r$d), PCfull), 5), "\n")
}

cat("\nPCAone, window based\n")
for(e in 1:7){
  set.seed(1); r <- winsvd(A, k = 2, epochs = e)
  cat("  ", e, "epochs           (", formatC(e, width=2), "passes,",
      formatC(r$updates, width=3), "updates ): dcor", round(dcor(r$u %*% diag(r$d), PCfull), 5), "\n")
}

In [ ]:
set.seed(1); h1 <- halko(A, k = 2, iter = 1)      # one power iteration  = 2 passes
set.seed(1); w1 <- winsvd(A, k = 2, epochs = 1)   # one window epoch     = 1 pass

par(mfrow = c(1,3))
plotPC(PCfull,                    "exact svd()")
plotPC(h1$u %*% diag(h1$d),       "Halko, 1 power iteration")
plotPC(w1$u %*% diag(w1$d),       "PCAone, 1 window epoch")

 - Compare the two tables at the same number of passes over the data. How many passes does Halko
   need to reach what one window epoch achieves?
 - Count the updates of $\Omega$ per pass in each algorithm. Where does the speed-up come from -
   doing less work, or doing the same work in a smarter order?
 - The first line of `winsvd` permutes the SNPs. Comment it out and rerun on data where the SNPs
   are ordered along a chromosome with linkage disequilibrium. What could go wrong with a window
   of physically neighbouring SNPs?
 - The window doubles until it covers the whole matrix, and the last epochs are ordinary power
   iterations. Why start small and end big rather than the other way round?
 - Each window position updates $Q$ from a *partial* sum $H$, so early in a sweep $Q$ is built
   from only a fraction of the genome. Why is that not a problem?
 - Both `winsvd` and `halko` end with `svd(t(Q) %*% A)` on a matrix with only $l$ rows. Why can
   we afford an exact SVD there?

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/em_algorithms/quiz/pca_em_rsvd.json")


 - Put the three numbers next to each other: called genotypes, dosage with the population
   frequency, and PCAngsd. Which step gives the biggest improvement, and which idea does that
   step correspond to?
 - Compare `pcangsd` with `emu` line by line. What is genuinely different, and what is the same?
 - At a site where an individual has no reads, the dosage is $2\Pi_{ij}$. In the first
   iteration that is $2f_j$, i.e. mean imputation. What is it in the last iteration?
 - The real `PCAngsd` also updates the allele frequency and estimates admixture proportions and
   selection statistics from $\Pi$. Why is $\Pi$ such a useful thing to have?
 - Increase the depth to 5X in `simGL` and rerun. How much do the three approaches differ then?

# Wrap up

Both algorithms fill in what they do not know with the expectation given the current
parameters, and then re-estimate the parameters from the completed data.

| | EMU | PCAngsd | PCAone |
|---|---|---|---|
| the problem | genotypes are missing | genotypes are uncertain | the matrix is too big for a full SVD |
| the data | $G$ with `NA` | genotype likelihoods $p(X\mid G)$ | the normalized genotypes $A$ |
| what is unknown | the missing genotypes | the true genotypes | the top $k$ subspace |
| E-step | predict the missing genotypes as $2\Pi_{ij}$ | dosage $\mathbb{E}[G\mid X,\Pi]$ at every entry | look at the data through the current subspace, $\Omega \leftarrow \text{orth}(A^{T}Q)$ |
| M-step | truncated SVD of the completed matrix | truncated SVD of the dosage matrix | re-estimate the subspace, $Q \leftarrow \text{orth}(A\Omega)$ |
| what stays fixed | the observed genotypes | nothing - every entry is updated | the data; only the window moves |
| the trick | use the PCA as the imputer | use the PCA as the prior | update $\Omega$ many times per pass over the data |

**Bonus**

 - EMU keeps the observed genotypes fixed while PCAngsd recomputes every entry. Explain why
   that difference is not really a difference. (hint: what is the dosage at a site with 30 reads?)
 - Both algorithms need $k$, the number of components. Suggest a way of choosing it from the
   data, and try it on the simulation above.
 - The `dcor` measure we used compares pairwise distances. Can you think of a case where it
   would say two PCA plots are identical when they are not?